Chronological Customer Trajectory Reconstruction

We will reorder transaction for a given customer by event time to combine events that happen during the samession in the correct order

In [1]:
from pathlib import Path
import json
import pandas as pd

# Resolve the project root whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'data_prep' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRANSCRIPT_PATH = RAW_DIR / 'transcript.json'
PORTFOLIO_PATH = RAW_DIR / 'portfolio.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Transcript path: {TRANSCRIPT_PATH}')

Project root: D:\SanJose\sem2\252_PROJECT
Transcript path: D:\SanJose\sem2\252_PROJECT\data\raw\transcript.json


In [2]:
# Load the line-delimited transcript and flatten the nested value object.
records = []
with TRANSCRIPT_PATH.open(encoding='utf-8') as stream:
    for line in stream:
        if not line.strip():
            continue
        event = json.loads(line)
        value = event.pop('value', {}) or {}
        event['offer_id'] = value.get('offer_id', value.get('offer id'))
        event['amount'] = value.get('amount')
        event['reward'] = value.get('reward')
        records.append(event)

events = pd.DataFrame(records)
events = events.sort_values(['person', 'time'], kind='stable').reset_index(drop=True)

print(f'Rows loaded: {len(events):,}')
print(f'Columns: {list(events.columns)}')

Rows loaded: 306,534
Columns: ['person', 'event', 'time', 'offer_id', 'amount', 'reward']


In [3]:
# First five chronological rows.
print('First five chronological event rows:')
display(events.head(5))

First five chronological event rows:


,person,event,time,offer_id,amount,reward
0,0009655768c64bdeb2e877511632db8f,offer received,168,5a8bc65990b245e5a138643cd4eb9837,NaN,NaN
1,0009655768c64bdeb2e877511632db8f,offer viewed,192,5a8bc65990b245e5a138643cd4eb9837,NaN,NaN
2,0009655768c64bdeb2e877511632db8f,transaction,228,None,22.16,NaN
3,0009655768c64bdeb2e877511632db8f,offer received,336,3f207df678b143eea3cee63160fa8bed,NaN,NaN
4,0009655768c64bdeb2e877511632db8f,offer viewed,372,3f207df678b143eea3cee63160fa8bed,NaN,NaN


In [4]:
# Join offer metadata while preserving every transcript event.
portfolio_records = []
with PORTFOLIO_PATH.open(encoding='utf-8') as stream:
    for line in stream:
        if line.strip():
            portfolio_records.append(json.loads(line))

portfolio = pd.DataFrame(portfolio_records).rename(columns={
    'id': 'offer_id',
    'reward': 'portfolio_reward',
})
events = events.merge(portfolio, on='offer_id', how='left', validate='many_to_one')

print(f'Offer events matched to portfolio metadata: {events["offer_type"].notna().sum():,}')

Offer events matched to portfolio metadata: 167,581


In [7]:
# First 10 chronological rows.
print('First five chronological event rows:')
display(events.head(10))

First five chronological event rows:


,person,event,time,offer_id,amount,reward,portfolio_reward,channels,difficulty,duration,offer_type
0,0009655768c64bdeb2e877511632db8f,offer received,168,5a8bc65990b245e5a138643cd4eb9837,NaN,NaN,0.0,"[email, mobile, social]",0.0,3.0,informational
1,0009655768c64bdeb2e877511632db8f,offer viewed,192,5a8bc65990b245e5a138643cd4eb9837,NaN,NaN,0.0,"[email, mobile, social]",0.0,3.0,informational
2,0009655768c64bdeb2e877511632db8f,transaction,228,None,22.16,NaN,NaN,NaN,NaN,NaN,NaN
3,0009655768c64bdeb2e877511632db8f,offer received,336,3f207df678b143eea3cee63160fa8bed,NaN,NaN,0.0,"[web, email, mobile]",0.0,4.0,informational
4,0009655768c64bdeb2e877511632db8f,offer viewed,372,3f207df678b143eea3cee63160fa8bed,NaN,NaN,0.0,"[web, email, mobile]",0.0,4.0,informational
5,0009655768c64bdeb2e877511632db8f,offer received,408,f19421c1d4aa40978ebb69ca19b0e20d,NaN,NaN,5.0,"[web, email, mobile, social]",5.0,5.0,bogo
6,0009655768c64bdeb2e877511632db8f,transaction,414,None,8.57,NaN,NaN,NaN,NaN,NaN,NaN
7,0009655768c64bdeb2e877511632db8f,offer completed,414,f19421c1d4aa40978ebb69ca19b0e20d,NaN,5.0,5.0,"[web, email, mobile, social]",5.0,5.0,bogo
8,0009655768c64bdeb2e877511632db8f,offer viewed,456,f19421c1d4aa40978ebb69ca19b0e20d,NaN,NaN,5.0,"[web, email, mobile, social]",5.0,5.0,bogo
9,0009655768c64bdeb2e877511632db8f,offer received,504,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,2.0,"[web, email, mobile, social]",10.0,10.0,discount


In [6]:
# Data format: shape, column types, and non-null counts.
print('Data format:')
print(f'Shape: {events.shape}')
print('\nColumn data types:')
print(events.dtypes)
print('\nNon-null counts:')
print(events.notna().sum())

Data format:
Shape: (306534, 11)

Column data types:
person               object
event                object
time                  int64
offer_id             object
amount              float64
reward              float64
portfolio_reward    float64
channels             object
difficulty          float64
duration            float64
offer_type           object
dtype: object

Non-null counts:
person              306534
event               306534
time                306534
offer_id            167581
amount              138953
reward               33579
portfolio_reward    167581
channels            167581
difficulty          167581
duration            167581
offer_type          167581
dtype: int64


In [8]:
# Next We will build one row per customer-offer pair from the event-level transcript.
# This creates a compact lifecycle table for downstream analysis or modeling.
offer_events = events[events['offer_id'].notna()].copy()
# Use numeric day values for lifecycle calculations such as time-to-view.
offer_events['event_time'] = offer_events['time'].astype(float)

# Keeping the first occurrence of each lifecycle event for each customer-offer.
offer_lifecycle = (
    offer_events.pivot_table(
        index=['person', 'offer_id'],
        columns='event',
        values='event_time',
        aggfunc='min',
    )
    .rename(columns={
        'offer received': 'received_time',
        'offer viewed': 'viewed_time',
        'offer completed': 'completed_time',
    })
    .reset_index()
)

# Now, attach one copy of the static offer metadata to each customer-offer row.
offer_metadata = (
    offer_events[['offer_id', 'offer_type', 'difficulty', 'duration', 'portfolio_reward']]
    .drop_duplicates('offer_id')
)
offer_trajectories = offer_lifecycle.merge(offer_metadata, on='offer_id', how='left', validate='many_to_one')
# Convert lifecycle timestamps into boolean outcomes and elapsed-time features.
offer_trajectories['completed'] = offer_trajectories['completed_time'].notna()
offer_trajectories['viewed'] = offer_trajectories['viewed_time'].notna()
offer_trajectories['days_to_view'] = offer_trajectories['viewed_time'] - offer_trajectories['received_time']
offer_trajectories['days_to_complete'] = offer_trajectories['completed_time'] - offer_trajectories['received_time']

# Sum transactions during each offer's active window. This is a time-window
# feature; overlapping offers may therefore observe the same transaction.
# Keep valid transaction events so we can measure spend during each offer window.
transactions = events.loc[events['event'].eq('transaction'), ['person', 'time', 'amount']].dropna(subset=['amount'])
transactions_by_person = {
    person: group[['time', 'amount']].to_numpy()
    for person, group in transactions.groupby('person')
}

# Index transactions by customer to avoid comparing an offer with other customers' events.
def window_transaction_amount(row):
    # The active window starts at receipt and lasts for the offer duration in days.
    person_transactions = transactions_by_person.get(row['person'])
    if person_transactions is None or pd.isna(row['received_time']):
        return 0.0
    window_end = row['received_time'] + row['duration']
    mask = (person_transactions[:, 0] >= row['received_time']) & (person_transactions[:, 0] <= window_end)
    return float(person_transactions[mask, 1].sum())

offer_trajectories['transaction_amount'] = offer_trajectories.apply(window_transaction_amount, axis=1)
offer_trajectories = offer_trajectories.sort_values(['person', 'received_time', 'offer_id'], kind='stable').reset_index(drop=True)

print(f'Offer trajectories: {len(offer_trajectories):,} rows')
print(f'Completed offers: {offer_trajectories["completed"].sum():,}')
print(f'Viewed offers: {offer_trajectories["viewed"].sum():,}')
display(offer_trajectories.head())


Offer trajectories: 63,288 rows
Completed offers: 28,996
Viewed offers: 49,135


,person,offer_id,completed_time,received_time,viewed_time,offer_type,difficulty,duration,portfolio_reward,completed,viewed,days_to_view,days_to_complete,transaction_amount
0,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,NaN,168.0,192.0,informational,0.0,3.0,0.0,False,True,24.0,NaN,0.00
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,NaN,336.0,372.0,informational,0.0,4.0,0.0,False,True,36.0,NaN,0.00
2,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,bogo,5.0,5.0,5.0,True,True,48.0,6.0,0.00
3,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,discount,10.0,10.0,2.0,True,True,36.0,24.0,0.00
4,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,NaN,discount,10.0,7.0,2.0,True,False,NaN,0.0,10.27


In [9]:
# Summarize the structure and completeness of the joined event dataframe.
# The shape is reported as (number_of_rows, number_of_columns).
print('Data format:')
print(f'Shape: {events.shape}')

# Show the pandas dtype assigned to each column, such as object, int64, or float64.
print('\nColumn data types:')
print(events.dtypes)

# Count non-missing values in each column. Counts below the total row count
# indicate fields that are only present for certain event types.
print('\nNon-null counts:')
print(events.notna().sum())

Data format:
Shape: (306534, 11)

Column data types:
person               object
event                object
time                  int64
offer_id             object
amount              float64
reward              float64
portfolio_reward    float64
channels             object
difficulty          float64
duration            float64
offer_type           object
dtype: object

Non-null counts:
person              306534
event               306534
time                306534
offer_id            167581
amount              138953
reward               33579
portfolio_reward    167581
channels            167581
difficulty          167581
duration            167581
offer_type          167581
dtype: int64
